# Integrating Weekly High-Frequency Data

In mixed-frequency Dynamic Factor Models, the mathematical tick is structurally rigid (Monthly base mapped to Quarterly outputs). 
However, we can safely incorporate **Weekly** data (like Initial Jobless Claims or Weekly Economic Indexes) by pre-aggregating them in pandas. This keeps the matrix algebra stationary while allowing the Kalman filter to update real-time off intra-month changes.

In [ ]:
import numpy as np
import pandas as pd
from dfm_sp import SpecConfig

### 1. Generating Raw Data
Let's simulate raw weekly releases, alongside standard monthly and quarterly targets.

In [2]:
# Fake Weekly Data
dates_weekly = pd.date_range(start='2020-01-01', end='2023-12-31', freq='W-FRI')
np.random.seed(42)
weekly_values = np.cumsum(np.random.normal(0, 1, size=len(dates_weekly))) + 100
weekly_series = pd.Series(weekly_values, index=dates_weekly, name="Weekly_Claims")

# Fake Monthly/Quarterly
dates_monthly = pd.date_range(start='2020-01-01', end='2023-12-01', freq='MS')
monthly_data = pd.Series(np.cumsum(np.random.normal(0, 2, size=len(dates_monthly))), index=dates_monthly, name="Industrial_Production")

dates_quarterly = pd.date_range(start='2020-01-01', end='2023-12-01', freq='QS-OCT')
quarterly_data = pd.Series(np.cumsum(np.random.normal(0, 5, size=len(dates_quarterly))), index=dates_quarterly, name="Real_GDP")

### 2. The Resample Step
We resample the weekly data dynamically to match `MS` (Month-Start). 

- `.mean()` : Good for **Flows** (e.g., Average Jobs claimed over the 4 weeks)
- `.last()` : Good for **Stocks** (e.g., Interest Rates exactly at end of month)

In [3]:
monthly_from_weekly = weekly_series.resample('MS').mean()

# Join heavily against our other matrices
df = pd.DataFrame(index=dates_monthly)
df = df.join(monthly_from_weekly)
df = df.join(monthly_data)
df = df.join(quarterly_data)

print("Aligned Matrix:")
display(df.head())

Aligned Matrix:


,Weekly_Claims,Industrial_Production,Real_GDP
2020-01-01,101.337097,7.705463,-3.538347
2020-02-01,103.511636,8.847244,NaN
2020-03-01,103.960674,11.118375,NaN
2020-04-01,100.052106,13.026379,-1.319250
2020-05-01,97.861659,14.329161,NaN


### 3. Loading Into DFM Specs
We map the newly formatted weekly data exactly as a standard `'m'` (Monthly) series in `SpecConfig`. The matrix transforms (e.g. `pch` or `dln` from `sp_transformations.py`) will automatically execute identically against it.

In [4]:
spec = SpecConfig(
    series_id=["W_CLAIMS", "IP_MONTHLY", "GDP_QUARTERLY"],
    series_name=["Initial Claims (Weekly Avg)", "Industrial Production", "Real GDP"],
    
    # Note index [0] is "m"
    frequency=["m", "m", "q"], 
    
    units=["Levels", "Levels", "Levels"],
    transformation=["lin", "lin", "lin"], 
    category=["Labor", "Production", "Output"],
    block_names=["Global", "Soft", "Real"],
    blocks_matrix=np.array([
        [1, 1, 0], # Weekly Proxy mapped onto Global and Soft
        [1, 0, 1], # Monthly IP mapped onto Global and Real
        [1, 0, 1], # Quarterly GDP mapped onto Global and Real
    ])
)

print(f"Target   : {spec.series_name[0]}")
print(f"Frequency: {spec.frequency[0]}")
print("Ready for feed into `get_with_options()`!")

Target   : Initial Claims (Weekly Avg)
Frequency: m
Ready for feed into `get_with_options()`!
